# Mini projekt

Niniejszym laboratorium rozpoczynamy część zajęć poświęconą budowaniu spójnego potoku masowego przetwarzania danych (tzw. mini projekt). Na kolejnych zajęciach będziemy rozbudowywali stworzony dziś moduł i dokładali do niego kolejne elementy.

Diagram planowanego systemu wygląda następująco:
![image](./assets/project_flow_chart.png)


## Laboratorium 8 – Implementacja rozproszonego mechanizmu pozyskiwania danych

Poniższe zadania realizują zakres Etapu I

### Zadanie 8.1: pozyskiwanie danych

Używając zdobytej dotychczas wiedzy, stwórz task Celery, którego zadaniem będzie pobieranie aktualnych wpisów z mikrobloga serwisu Wykop.pl (lub innego serwisu, który oferuje treści potrzebne do realizacji zadań - przeczytaj kilka kolejnych zadań zanim zdecydujesz się na konkretny portal i zweryfikuj czy potrzebne pola/metadane są tam dostępne).

Możesz:  
a) uzyskać dostęp do API: [dokumentacja](https://www.wykop.pl/dla-programistow/apiv2/)  
b) stworzyć mechanizm scrapowania

Pozyskaj dane z [najnowszych wpisów](https://www.wykop.pl/mikroblog/aktywne/) lub wpisów dotyczących [konkretnego tagu](https://www.wykop.pl/tag/wpisy/wroclaw/) - wybór tagu wedle uznania

Pozyskaj dane w formie pojedynczych wpisów. Zadbaj o zebranie:
* treści wpisu
* daty utworzenia
* nazwy autora wpisu 
* liczby plusów
* liczby odpowiedzi



### Zadanie 8.2: ciągłość procesu

Zadbaj o ciągłość pobierania danych:  
* ustal odpowiedni interwał pojawiania się nowych danych  
* użyj mechanizmu [celery-beat](https://docs.celeryproject.org/en/stable/userguide/periodic-tasks.html) by stworzyć mechanizm harmonogramowania tasków, który umożliwi pobieranie nowych wpisów


### Zadanie 8.3: monitoring

Dodaj możliwość monitorowania procesu pobierania danych:  
* dodaj do docker-compose instancję bazy danych, np. [Prometheus](https://prometheus.io/) lub [InfluxDB](https://www.influxdata.com/)
* dodaj do docker-compose instancję [Grafany](https://grafana.com/grafana/download?pg=get&plcmt=selfmanaged-box1-cta1&platform=docker) ([materiały z PDIOW](https://pwr-ai.github.io/przetwarzanie-danych-i-odkrywanie-wiedzy/laboratoria/lab6-produktyzacja.html#grafana-ladniejsze-wykresy-wiecej-mozliwosci))
* alternatywnie do w/w, możesz użyć [wersji chmurowej Grafany](https://grafana.com/get/?plcmt=top-nav&cta=downloads)
* zadbaj o zbieranie statystyk ze skryptu pobierającego dane. Przykłady zapisu danych w [Prometheusie](https://github.com/prometheus/client_python#exporting-to-a-pushgateway) i [Influxie](https://docs.influxdata.com/influxdb/v2.1/api-guide/client-libraries/python/)
* stwórz odpowiednie dashboardy ilustrujące proces zbierania danych. Stwórz *co najmniej* wykresy:
 - średniego czasu pobierania danych
 - ilości pobranych danych w czasie
 - histogram liczby plusów i odpowiedzi
 




## Laboratorium 9 – Implementacja mechanizmu czyszczenia i ekstrakcji cech

Niniejsze zadania dotyczą przygotowania danych wejściowych (zakres Etapu II) do modelu predykcji popularności wpisu na podstawie jego treści, który implementować będziemy na kolejnych zajęciach.  

### Zadanie 9.1: modelowanie danych

Zmodyfikuj zadanie pobierania danych:

a) stwórz model danych, modelujący pojedynczy wpis, zawierający co najmniej:
* tekst wpisu
* liczba plusów
* liczba komentarzy

Model danych będziemy przekazywać pomiędzy zadaniami i uaktualniać jego zawartość

b) przekaż każdy z pobranych wpisów oddzielnie poprzez kolejkę do kolejnego zadania. 

### Zadanie 9.2: sprawdzanie języka

Stwórz zadanie Celery, które będzie przyjmowało zamodelowany wpis z poprzedniego etapu.

Rozpoznaj język wpisu oraz odrzuć z przetwarzania wpisy w językach innych niż polski. Użyj dowolnej metody detekcji języka, np. [langdetect](https://pypi.org/project/langdetect/) lub [fasttext](https://fasttext.cc/docs/en/language-identification.html)

Zapisz statystyki detekcji języka do Grafany (w szczególności podział przetwarzanych języków)

Przekaż polskie wpisy do kolejnego etapu potoku

### Zadanie 9.3: budowanie reprezentacji wektorowej tekstu

Stwórz zadanie Celery, które będzie przyjmowało wpis z poprzedniego etapu.

Użyj dowolnej, działającej w języku polskim metody wektoryzacji, by zbudować reprezentację jego treści.

Możesz użyć:
* tzw. _word embeddingów_ - zadbaj o odpowiednią agregację wektorów. Przykładowe modele: [Word2Vec, GloVe, ELMO](https://github.com/sdadas/polish-nlp-resources#word2vec), [FastText](https://fasttext.cc/docs/en/crawl-vectors.html), [Flair](https://github.com/flairNLP/flair)
* modeli języka (ang. _language models_) - pamiętaj o odpowiednią agregację wektorów, jeśli to konieczne. Przykładowe modele pretrenowane do języka polskiego - [HerBERT](https://awesomeopensource.com/project/allegro/HerBERT), [Roberta](https://huggingface.co/clarin-pl/roberta-polish-kgr10) - lub modele wielojęzykowe np. [XLM-R](https://huggingface.co/xlm-roberta-base), [Labse](https://huggingface.co/sentence-transformers/LaBSE), [LASER](https://github.com/facebookresearch/LASER)

Zwektoryzuj treść wpisu do formy jednego wektora, uaktualnij model danych, a następnie przekaż go do kolejnego etapu.

### Zadanie 9.4: zapis danych

Dodaj do _docker compose_ instancję [MongoDB](https://www.mongodb.com/compatibility/docker). Upewnij się, że ustawienia persystencji danych mają odpowiednie wartości.

Stwórz task Celery, który będzie odbierał model danych z zadania wektoryzacji i zapisywał go w kolekcji Mongo.

### Zadanie 9.5: wizualizacja danych

Dodaj do _docker compose_ instancję [Redash](https://redash.io/help/open-source/setup#docker)

Podepnij bazę MongoDB jako źródło danych i przygotuj dashboard, który będzie przedstawiał chmurę słów (ang. _wordcloud_) dla najpopularniejszych i najmniej popularnych wpisów w ostatnim dniu - przyjmij odpowiednie progi.

## Laboratorium 10 – Uczenie i wybór modelu

Niniejsze zadanie dotyczą przygotowania potoku uczenia oraz wyboru modeli uczenia maszynowego przy użyciu platformy Spark (pySpark) – implementowane komponenty będą realizować zakres Etapu III. Alternatywnie do Sparka, można również wykorzystać Flinka.

### Zadanie 10.1: przygotowanie danych

Przygotuj kod, który pobierze dane z twojej bazy, a następnie zapisze je w formacie CSV (bądź innym który będziesz w stanie wczytać w kolejnych krokach).

### Zadanie 10.2: ładowanie i podział danych

[Wczytaj dane](https://sparkbyexamples.com/pyspark/pyspark-read-csv-file-into-dataframe/) za pomocą pySpark oraz dokonaj ich [podziału](https://spark.apache.org/docs/3.1.1/api/python/reference/api/pyspark.sql.DataFrame.randomSplit.html)

*  pySparka wykorzystujemy w trybie local, zadbaj o odpowiednią konfigurację

### Zadanie 10.3: zbuduj potok przetwarzania

Zbuduj [potok przetwarzania/pipeline](https://spark.apache.org/docs/latest/ml-pipeline.html#example-pipeline) który odpowiednio zmodyfikuje DataFrame do postaci akceptowanej przez pySpark ML, a następnie nauczy model. Umożliwij predykcję dla dowolnego tekstu oraz przeprowadź predykcję na danych testowych. Dokonaj ewaluacji w oparciu o dane testowe. Wykorzystaj [miary ewaluacji](https://spark.apache.org/docs/latest/mllib-evaluation-metrics.html#regression-model-evaluation) dla modeli regresji.

- dane musimy sprowadzić do postaci tabeli o kolumnach "features" i "label"
- na podstawie cech chcemy przewidywać ilość plusów (lub inną wybraną cechę/metadaną)
- wykorzystujemy LinearRegression
- możemy wprost wykorzystać zwektoryzowany tekst
- możemy zwektoryzować dodatkowe cechy za pomocą odpowiednich narzędzi \[[1](https://spark.apache.org/docs/latest/ml-features.html#featurehasher), [2](https://spark.apache.org/docs/latest/ml-features.html#onehotencoder), [3](https://spark.apache.org/docs/latest/ml-features.html#stringindexer)\] (dodatkowe punkty za dodatkowe atrybuty)
- dodatkowo możemy [rozbudować](https://spark.apache.org/docs/latest/ml-features.html#vectorassembler) zwektroryzowany tekst o dodatkowe zwektoryzowane cechy

### Zadanie 10.4: dobór parametrów

Stwórz kolejny potok przetwarząnia który dobierze parametry modelu regresji za pomocą podziału na [zbiór uczący i walidacyjny](https://spark.apache.org/docs/latest/ml-tuning.html#train-validation-split)

* piepline może być traktowany jako estymator, przez co może być przekazany do ```TrainValidationSplit```
* dokonaj ewaluacji najlepszego modelu na danych testowych
* dokonaj ewaluacji modelu na danych wprowadzanych "z palca" (tekst jako zmienna w kodzie)


## Laboratorium 12 – Uruchomienie opracowanej metody z wykorzystaniem środowiska Kubernetes

Niniejsze zadania realizują zakres Etapu IV - tzn. wykorzystując poznane narzędzie Helm oraz platformę Kubernetes opracujemy chart Helmowy dla naszego systemu.

**Uwaga 1:** Przed przystąpieniem do tej części, zrealizuj najpierw laboratorium dotyczące wstępu do Kubernetesa!

**Uwaga 2:** W poniższych zadaniach kompletnie pomijamy komponenty z Etapu III

### Zadanie 12.1: Przygotowanie charta Helmowego

a) Sprawdź, które komponenty opracowanej aplikacji (`service` w Docker-Compose) są dostępne jako gotowe charty Helmowe.

b) Dla pozostałych komponentów przygotuj obrazy Dockerowe - m.in. mechanizm pobierania danych z Etapu I oraz Detekcja języka, Wektoryzacja tekstu i zapis do bazy danych z Etapu II. Następnie wybierz jedną z opcji:
- A: udostępnij obrazy na platformie Docker Hub
- B: zbuduj obrazy na maszynach typu worker w ramach, utworzonego podczas ostatnich zajęć, klastra K3s

c) W repozytorium utwórz chart Helmowy dla naszej aplikacji:
- dodaj gotowe charty komponentów z punktu a)
- dla każdego komponentu z punktu b) utwórz odpowiednie szablony Helmowe (deployment, service, configmap itd. - w zależności od potrzeb)
- zadbaj o wolumeny danych (dla uproszczenia użyj `hostPath`, ale pamiętaj, że w rzeczywistych zastosowaniach należy użyć odpowiedniej Storage Classy, np. EBS, Cinder, NFS)


d) Do odpowiednich komponentów dodaj próbki liveness and readiness, które pozwolą sprawdzić Kubernetesowi czy aplikacja nadal działa poprawnie.

### Zadanie 12.2: Wdrożenie aplikacji
- Zainstaluj przygotowany chart Helmowy w klastrze K3s
- Sprawdź czy wszystko działa poprawnie
- Zmień wybraną wartość w pliku `values.yaml` (np. liczba replik komponentu detekcji języka)
- Przeprowadź rolling update
- Sprawdź czy zmiana została poprawnie zaaplikowana
- Odwróć zmiany i wróć do poprzedniej wersji aplikacji (rollback)

### Zadanie 12.3: Poprawność działania aplikacji
- Sprawdź, że dane są zbierane na bieżąco
- Pokaż, że Grafana działa i wyświetla dane na bieżąco
- Pokaż, że Redash działa i wyświetla dane na bieżąco

## Laboratorium 13 – Udostępnienie opracowanego modelu za pomocą gRPC

Niniejsze zadania rozpoczynają prace nad Etapem IV.

Wykorzystując język Python udostępnimy opracowany model za pomocą gRPC. Zapoznaj się z poniższymi źródłami:

* [Basics](https://grpc.io/docs/languages/python/basics/)
* [API Docs](https://grpc.github.io/grpc/python/)

### Zadanie 13.0: Zapoznaj się z przykładowym projektem
Zapoznaj się z przykładowym projektem (katalog `lab13/`) oraz jego readme (znajdują się tam potrzebne komendy). Co dzieje się w projekcie? Jaki kod jest generowany oraz jak jest wykorzystywany? 

### Zadanie 13.1: Przygotowanie definicji API
Przygotuj plik proto, zawierający definicję odpowiedniego serwisu wraz z metodą oraz odpowiednimi typami wejściowymi i wyjściowymi które pozwolą na ocenę ilości plusów które otrzyma wpis o zadanej treści. Pamiętaj o odpowiedniej modyfikacji kroku POE w pliku pyproject.toml (budowanie odpowiedniego pliku proto, domyślnie jest to sample) 

### Zadanie 13.2: Implementacja serwera oraz klienta
Zaimplementuj serwer oraz klienta wykorzystując wygenerowane definicje. Przygotuj przykładowe wywołania metod serwera za pomocą klienta

### Zadanie 13.3: Implementacja serwera oraz klienta
Przygotuj obraz Dockerowy oraz chart Helmowy, a następnie uruchom rozwiązanie na klasterze k3s.

## Laboratorium 14 – Udostępnienie aplikacji SPA do predykcji popularności

Niniejsze zadania kończą realizację Etapu IV.

Wykorzystując język Python udostępnimy opracowany model jako aplikację SPA za pomocą GraphQL. Zapoznaj się z poniższymi źródłami:

* [Backend - Ariadne](https://ariadnegraphql.org/)
* [Frontend - Apollo](https://www.apollographql.com/docs/react/)

### Zadanie 14.0: Zapoznaj się z przykładowym projektem
Zapoznaj się z przykładowym projektem (katalog `lab14/`) oraz jego readme (znajdują się tam potrzebne komendy). Co dzieje się w projekcie? Jak odbywa się komunikacja? Czym jest schemat GraphQL? 

### Zadanie 14.1: Przygotowanie definicji API
Zaktualizuj plik schema, stwórz odpowiednie zapytanie oraz zdefiniuj odpowiednie typy dla zadania predykcji popularności.

### Zadanie 14.2: Implementacja serwera oraz SPA
Zaimplementuj resolver dla swojego zapytania (wykorzystaj klienta gRPC z poprzedniej listy), stwórz frontend do jego wywoływania. 

### Zadanie 14.3: Uruchomienie
Przygotuj obraz Dockerowy oraz chart Helmowy, a następnie uruchom rozwiązanie na klasterze k3s.